# 01｜用 Python 理解一份陌生資料（對應課程 M3）

**本 Notebook 的目標**：不靠 AI，用 pandas 完成 M3「資料健檢」的四件事：

1. 每張表的列數、欄位、型態、缺值
2. 每張表的**粒度**（一列代表什麼）與主鍵是否唯一
3. 表與表的關聯（一對一或一對多），以及有沒有對不上的鍵
4. 依「資料品質六面向」逐項檢查，整理成**陷阱紀錄表**

> **AI 做法 vs 傳統做法**
> 在課堂上，你用一段提示請 Claude 或 ChatGPT 做健檢；在這裡，每一項檢查都是一段你看得懂、可以重跑的程式。
> 兩者結果應該一致，這本身就是一種「對帳」。

**先備**：Python 3.10 以上，`pip install pandas numpy matplotlib scipy jupyter`

## 0. 環境準備

`islemart_lib.py` 放在 Notebook 同一個資料夾，裡面有讀資料與設定中文字型的共用函式。
資料預設從 `../模擬資料_IsleMart/學員版/islemart_data` 讀取；放在別處時，設定環境變數 `ISLEMART_DATA` 即可。

In [ ]:
import sys
import pandas as pd
import numpy as np
import islemart_lib as L

print("Python", sys.version.split()[0], "｜ pandas", pd.__version__)
print("資料夾：", L.find_data_dir())
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. 讀進 9 張表，先看列數與欄位

`load_all()` 幫我們做了三件容易出錯的事：
- 用 `utf-8-sig` 讀檔，去掉檔頭的 BOM（否則第一個欄位名稱會多一個看不見的字元）
- 郵遞區號當**文字**讀，避免變成數字
- 時間欄位轉成 `datetime`

In [ ]:
T = L.load_all()

overview = pd.DataFrame({
    "列數": {k: len(v) for k, v in T.items()},
    "欄位數": {k: v.shape[1] for k, v in T.items()},
})
overview

In [ ]:
T["orders"].head(3)

## 2. 每張表的「體檢報告」

寫一個小函式 `profile()`，一次列出每個欄位的型態、缺值、唯一值數量與範例。
這就是 AI 產出的「資料字典」的原型。

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "型態": df.dtypes.astype(str),
        "缺值數": df.isna().sum(),
        "缺值%": (df.isna().mean() * 100).round(2),
        "唯一值數": df.nunique(),
        "範例": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

profile(T["orders"])

In [ ]:
# 其他表也可以照樣檢查，例如商品表
profile(T["products"])

**觀察**：`orders` 的送達時間有缺值、`products` 的類別有缺值。
缺值本身不一定是錯：取消的訂單本來就沒有送達時間。下一步要分辨「合理的缺」與「不該缺」。

## 3. 粒度與主鍵：一列到底代表什麼？

檢查方法：如果某個（或某組）欄位是主鍵，它在整張表裡**不能重複**。

In [ ]:
def check_key(df: pd.DataFrame, cols: list[str]) -> dict:
    dup = df.duplicated(subset=cols, keep=False)
    return {"鍵": " + ".join(cols), "列數": len(df),
            "不重複鍵數": len(df.drop_duplicates(cols)), "重複列數": int(dup.sum()),
            "是否唯一": not dup.any()}

key_checks = pd.DataFrame([
    check_key(T["orders"], ["order_id"]),
    check_key(T["order_items"], ["order_id", "order_item_id"]),
    check_key(T["order_payments"], ["order_id", "payment_sequential"]),
    check_key(T["order_reviews"], ["review_id"]),
    check_key(T["order_reviews"], ["order_id"]),
    check_key(T["customers"], ["customer_id"]),
    check_key(T["customers"], ["customer_unique_id"]),
    check_key(T["sellers"], ["seller_id"]),
    check_key(T["products"], ["product_id"]),
    check_key(T["geolocation"], ["geolocation_zip_code_prefix"]),
])
key_checks

**解讀**

- `order_reviews`：`review_id` 竟然有重複 → 有**完全重複的列**；`order_id` 也重複 → 有訂單收到**兩則評論**。
- `customers`：`customer_id` 唯一，但 `customer_unique_id` 不唯一。
  也就是說，**customers 的一列是「一筆訂單的顧客身分」，不是「一個人」**。要算回購率，必須用 `customer_unique_id`。
- `geolocation`：同一個郵遞區號有很多列座標，直接 JOIN 會讓資料暴增。

## 4. 表與表的關聯：一對一還是一對多？

`relationship()` 計算兩件事：
1. 每個鍵在左表、右表**最多出現幾次**（判斷一對一、一對多）
2. 有多少鍵在另一邊**找不到對應**（參照完整性）

In [ ]:
def relationship(left, right, lkey, rkey=None, lname="左", rname="右"):
    rkey = rkey or lkey
    lmax = left[lkey].value_counts().max()
    rmax = right[rkey].value_counts().max()
    kind = {(True, True): "一對一", (True, False): "一對多",
            (False, True): "多對一", (False, False): "多對多"}[(lmax == 1, rmax == 1)]
    lset, rset = set(left[lkey].dropna()), set(right[rkey].dropna())
    return {"關聯": f"{lname}.{lkey} → {rname}.{rkey}", "類型": kind,
            f"左邊找不到右邊": len(lset - rset), f"右邊找不到左邊": len(rset - lset)}

rels = pd.DataFrame([
    relationship(T["customers"], T["orders"], "customer_id", lname="customers", rname="orders"),
    relationship(T["orders"], T["order_items"], "order_id", lname="orders", rname="order_items"),
    relationship(T["orders"], T["order_payments"], "order_id", lname="orders", rname="order_payments"),
    relationship(T["orders"], T["order_reviews"], "order_id", lname="orders", rname="order_reviews"),
    relationship(T["products"], T["order_items"], "product_id", lname="products", rname="order_items"),
    relationship(T["sellers"], T["order_items"], "seller_id", lname="sellers", rname="order_items"),
    relationship(T["category_translation"], T["products"], "product_category_name",
                 lname="translation", rname="products"),
])
rels

**解讀**

- `orders` 對 `order_items`、`order_payments`、`order_reviews` 都是**一對多** → 合併時要先彙總（Notebook 02 會示範）。
- 「右邊找不到左邊」：有些訂單沒有評論（合理），有些商品類別在翻譯表裡**找不到**（不合理）。

In [ ]:
# 哪些類別沒有英文翻譯？
missing_cat = set(T["products"]["product_category_name"].dropna()) - set(T["category_translation"]["product_category_name"])
missing_cat

## 5. 資料品質六面向，逐項檢查

對照講義 M3「資料品質的六個面向」：完整性、唯一性、一致性、合理性、參照完整性、時效性。
每找到一個問題，就記到 `issues` 清單，最後整理成陷阱紀錄表。

In [ ]:
issues = []

def log_issue(面向, 描述, 表, 筆數, 影響):
    issues.append({"面向": 面向, "描述": 描述, "表": 表, "筆數": int(筆數), "會害你算錯什麼": 影響})
    print(f"[{面向}] {描述}：{筆數:,} 筆")

o = T["orders"]

### 5.1 完整性：缺值合不合理？

把「送達時間缺值」依訂單狀態拆開看。取消、處理中的訂單沒有送達時間是合理的；**已送達卻沒有送達時間**就不合理。

In [ ]:
pd.crosstab(o["order_status"], o["order_delivered_customer_date"].isna(),
            rownames=["狀態"], colnames=["送達時間缺值"])

In [ ]:
n = ((o["order_status"] == "delivered") & o["order_delivered_customer_date"].isna()).sum()
log_issue("完整性", "狀態為 delivered 卻沒有送達時間", "orders", n, "配送天數被忽略或計算報錯")

n = T["products"]["product_category_name"].isna().sum()
log_issue("完整性", "商品沒有類別", "products", n, "類別分析出現空白組")

### 5.2 唯一性：重複的列與重複的評論

In [ ]:
rv = T["order_reviews"]
log_issue("唯一性", "完全重複的評論列", "order_reviews", rv.duplicated().sum(), "平均評分被重複計算")

multi = rv.drop_duplicates().groupby("order_id").size()
log_issue("唯一性", "同一訂單有兩則以上評論", "order_reviews", (multi > 1).sum(), "評論與訂單 JOIN 後列數膨脹")

c = T["customers"]
log_issue("唯一性", "customer_id 每單一個，同一人要看 customer_unique_id", "customers",
          len(c) - c["customer_unique_id"].nunique(), "顧客數高估、回購率算成 0%")

### 5.3 一致性：同一件事有沒有不同寫法？

把縣市名稱列出來看，就會發現「台北市」與「臺北市」同時存在。

In [ ]:
c["customer_county"].value_counts().sort_index().head(12)

In [ ]:
log_issue("一致性", "縣市名稱「台／臺」混用", "customers",
          c["customer_county"].str.contains("臺").sum(), "同一縣市被拆成兩組")

### 5.4 合理性：時間先後、金額範圍、狀態矛盾

**價格異常**用「相對」標準判斷，而不是隨便訂一個門檻：
單價 ≥ 同類別中位數的 20 倍，就視為異常（例如多打了兩個 0）。
固定門檻（例如 5 萬元）會漏掉便宜商品被打錯的情況。

In [ ]:
it = T["order_items"]
it["price"].describe().round(1)

In [ ]:
outlier = L.flag_price_outliers(it, T["products"])
print("固定門檻 > 50,000 抓到：", (it["price"] > 50_000).sum(), "列")
print("相對標準（類別中位數 20 倍）抓到：", outlier.sum(), "列")
it.loc[outlier, ["order_id", "product_id", "price"]].head()

In [ ]:
log_issue("合理性", "單價異常（約為正常價格的 100 倍）", "order_items", outlier.sum(), "營收高估、平均單價失真")

n = (o["order_delivered_customer_date"] < o["order_delivered_carrier_date"]).sum()
log_issue("合理性", "送達時間早於交給物流的時間", "orders", n, "配送天數出現負值或過短")

n = ((o["order_status"] == "canceled") & o["order_delivered_customer_date"].notna()).sum()
log_issue("合理性", "已取消卻有送達時間", "orders", n, "狀態與時間戳記矛盾")

**付款與明細對不上**：每筆訂單的付款總額，應該等於商品售價＋運費。

In [ ]:
pay = T["order_payments"].groupby("order_id")["payment_value"].sum()
items_total = it.assign(t=it["price"] + it["freight_value"]).groupby("order_id")["t"].sum()
diff = (pay - items_total).dropna()
log_issue("合理性", "付款總額與明細加總不符", "order_payments", (diff.abs() > 1).sum(),
          "用付款或用明細算營收，答案不同")

multi_pay = T["order_payments"].groupby("order_id").size()
log_issue("合理性", "一筆訂單有多列付款（折價券＋主要付款）", "order_payments", (multi_pay > 1).sum(),
          "直接 JOIN 會重複計算")

**運費只記在第一列**：多品項訂單裡，第 2 列以後的運費都是 0。

In [ ]:
it.groupby(it["order_item_id"] == 1)["freight_value"].mean().rename({True: "第 1 列", False: "第 2 列以後"})

In [ ]:
log_issue("合理性", "運費只記在訂單第一列", "order_items",
          (it["order_item_id"] > 1).sum(), "用「平均每列運費」會低估")

### 5.5 參照完整性：翻譯表漏項、地理表一對多

In [ ]:
log_issue("參照完整性", f"翻譯表缺少類別：{'、'.join(sorted(missing_cat))}", "category_translation",
          len(missing_cat), "用英文類別分析時這些類別消失")

geo_dup = T["geolocation"].groupby("geolocation_zip_code_prefix").size()
log_issue("參照完整性", "同一郵遞區號有多列座標", "geolocation", (geo_dup > 1).sum(),
          "直接 JOIN 地理表會讓訂單列數暴增")

### 5.6 時效性：資料最後幾週完整嗎？

看每週訂單中，有多少比例還停在 `shipped`（已出貨、未送達）。

In [ ]:
weekly = (o.assign(week=o["order_purchase_timestamp"].dt.to_period("W").dt.start_time,
                   shipped=o["order_status"].eq("shipped"))
            .groupby("week")["shipped"].mean().mul(100).round(1))
weekly.tail(6)

In [ ]:
n = ((o["order_purchase_timestamp"] >= "2025-12-22") & (o["order_status"] == "shipped")).sum()
log_issue("時效性", "最後 10 天約一半訂單仍在途中（shipped）", "orders", n,
          "月趨勢最後一個月看起來「下滑」")

## 6. 陷阱紀錄表

把上面所有發現整理成一張表，並存成 CSV。這就是講義 M3 練習表的「程式版」。

In [ ]:
trap_log = pd.DataFrame(issues)
trap_log.index = trap_log.index + 1
trap_log

In [ ]:
trap_log.to_csv("trap_log.csv", encoding="utf-8-sig")
print(f"共記錄 {len(trap_log)} 項，已存成 trap_log.csv")

## 小結

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| 一段提示請 AI 做健檢 | `profile()`、`check_key()`、`relationship()` 三個函式 |
| AI 列出它發現的問題 | 依六面向逐項寫檢查，記到 `issues` |
| 追問「業務上不合理的地方」 | 用業務邏輯寫規則：狀態 vs 時間、付款 vs 明細、運費位置 |

**練習**：把你在課堂上請 AI 做的健檢結果，和這張表逐項比對。AI 漏了哪幾項？這張表又漏了哪幾項？